# INV-M365-CM — Persona-Action Execution Reuse v1

## Purpose
Reduce the mapped workforce graph into predecessor-certified reuse versus remaining live-proof scope for `G4`.

## Lemma Mapping
- `L87_m365_persona_action_execution_reuse_v1`

## Invariant Support
- `invariants/lemmas/L87_m365_persona_action_execution_reuse_v1.yaml`

## Expected Results
- `33` reusable green unique aliases
- `24` reusable fenced unique aliases
- `1` reusable approval-gated alias
- `1` reusable actor-tier-gated alias
- `56` remaining mapped aliases and `56` remaining mapped pairs requiring live proof in `G4`


In [ ]:
from __future__ import annotations
import ast
import json
from collections import defaultdict
from pathlib import Path
import sys
import yaml

repo = Path.cwd()
if not (repo / "registry/persona_registry_v2.yaml").exists():
    repo = Path("/Users/smarthaus/Projects/GitHub/M365")
sys.path.insert(0, str(repo / "src"))
from smarthaus_common.executor_routing import executor_route_for_action

root = repo / "registry"
preg = yaml.safe_load((root / "persona_registry_v2.yaml").read_text())
direct = json.loads((repo / "artifacts/diagnostics/m365_direct_full_surface_certification.json").read_text())
source = (repo / "src/ops_adapter/actions.py").read_text()
mod = ast.parse(source)
func = next(node for node in mod.body if isinstance(node, ast.AsyncFunctionDef) and node.name == "_execute_impl")

def parse_test(expr):
    agents = None
    actions = None
    if isinstance(expr, ast.Compare) and isinstance(expr.left, ast.Name) and len(expr.ops) == 1:
        name = expr.left.id
        comp = expr.comparators[0]
        vals = []
        if isinstance(comp, ast.Constant) and isinstance(comp.value, str):
            vals = [comp.value]
        elif isinstance(comp, ast.Tuple):
            vals = [elt.value for elt in comp.elts if isinstance(elt, ast.Constant) and isinstance(elt.value, str)]
        if isinstance(expr.ops[0], (ast.Eq, ast.In)) and vals:
            if name == "agent":
                agents = vals
            elif name == "action":
                actions = vals
    elif isinstance(expr, ast.BoolOp) and isinstance(expr.op, ast.And):
        for value in expr.values:
            a, b = parse_test(value)
            if a:
                agents = a if agents is None else sorted(set(agents) | set(a))
            if b:
                actions = b if actions is None else sorted(set(actions) | set(b))
    return agents, actions

def body_profile(stmts):
    body_has_await = any(isinstance(n, ast.Await) for stmt in stmts for n in ast.walk(stmt))
    first_return_kind = "none"
    for stmt in stmts:
        if isinstance(stmt, ast.Return):
            val = stmt.value
            if isinstance(val, ast.Await):
                first_return_kind = "await"
            elif isinstance(val, ast.Call):
                first_return_kind = "call"
            elif isinstance(val, (ast.Dict, ast.List, ast.Constant, ast.JoinedStr)):
                first_return_kind = "literal"
            elif isinstance(val, ast.Name):
                first_return_kind = "name"
            else:
                first_return_kind = type(val).__name__
            break
    return body_has_await, first_return_kind

rows = []
def walk(nodes, current_agents=None):
    for node in nodes:
        if isinstance(node, ast.If):
            agents, actions = parse_test(node.test)
            next_agents = current_agents
            if agents and not actions:
                next_agents = agents
            if actions and (agents or current_agents):
                final_agents = agents or current_agents or []
                body_has_await, first_return_kind = body_profile(node.body)
                rows.append({
                    "agents": list(final_agents),
                    "actions": list(actions),
                    "body_has_await": body_has_await,
                    "first_return_kind": first_return_kind,
                })
            walk(node.body, next_agents)
            walk(node.orelse, current_agents)
walk(func.body)

explicit_pairs = set()
pure_literal_pairs = set()
for row in rows:
    for agent in row["agents"]:
        for action in row["actions"]:
            explicit_pairs.add((agent, action))
            if row["first_return_kind"] == "literal" and not row["body_has_await"]:
                pure_literal_pairs.add((agent, action))

alias_map = defaultdict(set)
for path in root.glob("*_expansion_v2.yaml"):
    data = yaml.safe_load(path.read_text()) or {}
    for action, meta in (data.get("supported_actions") or {}).items():
        if isinstance(meta, dict) and meta.get("agent_alias"):
            alias_map[str(meta["agent_alias"])].add(str(action))
recipes = yaml.safe_load((root / "cross_workload_automation_recipes_v2.yaml").read_text()) or {}
for action, meta in (recipes.get("catalog_actions") or {}).items():
    if isinstance(meta, dict) and meta.get("agent_alias"):
        alias_map[str(meta["agent_alias"])].add(str(action))

known_stub_pairs = {(agent, action) for agent, actions in direct["legacy_stub_inventory"]["by_agent"].items() for action in actions}
expanded_stub_pairs = known_stub_pairs | pure_literal_pairs
non_stub_explicit_pairs = explicit_pairs - expanded_stub_pairs

pair_status = {}
for persona_id, persona in preg["personas"].items():
    status = persona["status"]
    for action in persona.get("allowed_actions", []):
        if status != "active":
            pair_status[(persona_id, action)] = "fenced"
            continue
        if action in alias_map:
            pair_status[(persona_id, action)] = "mapped"
            continue
        if (persona_id, action) in expanded_stub_pairs:
            pair_status[(persona_id, action)] = "legacy-stubbed"
            continue
        if (persona_id, action) in non_stub_explicit_pairs:
            pair_status[(persona_id, action)] = "mapped"
            continue
        try:
            executor_route_for_action(persona_id, action)
        except Exception:
            pair_status[(persona_id, action)] = "orphaned"
        else:
            pair_status[(persona_id, action)] = "dead-routed"

green_actions = set()
for group in [direct["f2_read_certification"]["certified_green_actions"], direct["f3_mutation_approval_certification"]["certified_green_actions"]]:
    for values in group.values():
        green_actions.update(values)
approval_aliases = set(direct["f3_mutation_approval_certification"]["approval_boundary_certification"]["pending_approval_actions"])
actor_tier_aliases = {"users.disable"}
fenced_actions = set()
for group in direct["f2_read_certification"]["fenced_actions"].values():
    fenced_actions.update(group["actions"])
fenced_actions.update(direct["f3_mutation_approval_certification"]["fenced_actions"]["planner_permissions"]["actions"])
for group in direct["f4_final_support_matrix"]["fenced_breakdown"]["f4_additional_fenced_write_groups"].values():
    fenced_actions.update(group["actions"])

mapped_pairs = []
for (persona_id, action), status in sorted(pair_status.items()):
    if status != "mapped":
        continue
    if action in approval_aliases:
        reuse = "approval-gated"
    elif action in actor_tier_aliases:
        reuse = "actor-tier-gated"
    elif action in alias_map:
        direct_actions = alias_map[action]
        if direct_actions & green_actions:
            reuse = "green"
        elif direct_actions & fenced_actions:
            reuse = "fenced"
        else:
            reuse = "requires-live-proof"
    else:
        reuse = "requires-live-proof"
    mapped_pairs.append({"persona_id": persona_id, "action": action, "reuse": reuse})

pair_counts = defaultdict(int)
for row in mapped_pairs:
    pair_counts[row["reuse"]] += 1

alias_reuse = defaultdict(set)
for row in mapped_pairs:
    alias_reuse[row["action"]].add(row["reuse"])
unique_alias_groups = defaultdict(list)
for action, statuses in sorted(alias_reuse.items()):
    if statuses == {"green"}:
        unique_alias_groups["green"].append(action)
    elif statuses == {"approval-gated"}:
        unique_alias_groups["approval-gated"].append(action)
    elif statuses == {"actor-tier-gated"}:
        unique_alias_groups["actor-tier-gated"].append(action)
    elif statuses == {"fenced"}:
        unique_alias_groups["fenced"].append(action)
    elif statuses == {"requires-live-proof"}:
        unique_alias_groups["requires-live-proof"].append(action)
    else:
        unique_alias_groups["mixed-reuse"].append({"action": action, "statuses": sorted(statuses)})

summary = {
    "reusable_unique_aliases": {
        "green": len(unique_alias_groups["green"]),
        "approval-gated": len(unique_alias_groups["approval-gated"]),
        "actor-tier-gated": len(unique_alias_groups["actor-tier-gated"]),
        "fenced": len(unique_alias_groups["fenced"]),
    },
    "reusable_pair_counts": dict(sorted(pair_counts.items())),
    "remaining_unique_aliases_requiring_live_proof_total": len(unique_alias_groups["requires-live-proof"]),
    "remaining_mapped_pairs_requiring_live_proof_total": sum(1 for row in mapped_pairs if row["reuse"] == "requires-live-proof"),
    "mixed_reuse_aliases_total": len(unique_alias_groups["mixed-reuse"]),
}
print(json.dumps(summary, indent=2))
assert summary["reusable_unique_aliases"] == {"green": 33, "approval-gated": 1, "actor-tier-gated": 1, "fenced": 24}
assert summary["reusable_pair_counts"] == {"actor-tier-gated": 1, "approval-gated": 1, "fenced": 72, "green": 135, "requires-live-proof": 56}
assert summary["remaining_unique_aliases_requiring_live_proof_total"] == 56
assert summary["remaining_mapped_pairs_requiring_live_proof_total"] == 56
assert summary["mixed_reuse_aliases_total"] == 0
